# Build a Dependency-Freshness Checker

Companion notebook for the [Build a Dependency-Freshness Checker](https://pyda-course.online/docs/projects/dependency-freshness-checker) Real-World Project.

No API key needed — this uses PyPI's free, public JSON API. Since a hosted notebook session has no real project folder of yours to point at, this notebook checks a sample `pyproject.toml` embedded below instead of a real file on disk. The local `uv`-based version (see the project's `examples/dependency-freshness-checker/` folder) can point at any real `pyproject.toml` on your machine.

In [ ]:
!pip install -q requests packaging

## Step 1: A sample `pyproject.toml`

In the real, local version of this project you'd read this from an actual file with `tomllib`. Here, we embed the same kind of content as a string and parse it the same way, so the rest of the logic is identical.

In [ ]:
import tomllib

SAMPLE_PYPROJECT = """
[project]
name = "sample-project"
version = "0.1.0"
dependencies = [
    "requests>=2.31",
    "packaging>=24.0",
    "flask==2.0.0",
    "not-a-real-package-xyz>=1.0",
]
"""


def load_dependencies(toml_text: str) -> list[str]:
    """Same logic as parse_deps.py's load_dependencies, just parsing a string
    instead of opening a file from disk."""
    data = tomllib.loads(toml_text)
    return data.get("project", {}).get("dependencies", [])


deps = load_dependencies(SAMPLE_PYPROJECT)
deps

## Step 2: Look up each package's current version on PyPI

In [ ]:
import re

import requests


def parse_package_name(specifier: str) -> str:
    """Extract just the package name from a specifier like 'requests>=2.31'
    or 'requests[socks]==2.31.0'."""
    match = re.match(r"^[A-Za-z0-9_.-]+", specifier.strip())
    if not match:
        raise ValueError(f"Could not parse a package name from {specifier!r}")
    return match.group(0)


def get_latest_version(package_name: str) -> str | None:
    """Query PyPI's public JSON API for a package's current published
    version. Returns None if the package isn't found."""
    url = f"https://pypi.org/pypi/{package_name}/json"
    response = requests.get(url, timeout=10)
    if response.status_code == 404:
        return None
    response.raise_for_status()
    return response.json()["info"]["version"]


for specifier in deps:
    name = parse_package_name(specifier)
    print(f"{name}: latest is {get_latest_version(name)!r}")

## Step 3: Compare versions correctly

Naive string comparison breaks (`"2.10" < "2.9"` as plain text!) — `packaging.version.Version` parses each part as a real number instead, the same library `pip` itself uses internally.

In [ ]:
from packaging.version import InvalidVersion, Version


def is_outdated(current: str, latest: str) -> bool | None:
    """Returns None (not True/False) if either string isn't a version
    packaging can parse -- e.g. a git URL used as a 'version'."""
    try:
        return Version(current) < Version(latest)
    except InvalidVersion:
        return None


print(is_outdated("2.9.0", "2.10.0"))  # True -- real semantic comparison, not string comparison
print(is_outdated("2.10.0", "2.9.0"))  # False
print(is_outdated("not-a-version", "2.10.0"))  # None -- can't compare

## Step 4: Put it together into a real freshness report

In [ ]:
from dataclasses import dataclass


@dataclass
class DependencyStatus:
    name: str
    current_specifier: str
    latest: str | None
    outdated: bool | None


def build_report(specifiers: list[str]) -> list[DependencyStatus]:
    report = []
    for specifier in specifiers:
        name = parse_package_name(specifier)
        latest = get_latest_version(name)
        pinned = specifier[len(name):].lstrip(">=<~! ")
        outdated = is_outdated(pinned, latest) if pinned and latest else None
        report.append(DependencyStatus(name, specifier, latest, outdated))
    return report


def print_report(report: list[DependencyStatus]) -> None:
    outdated = [d for d in report if d.outdated is True]
    fresh = [d for d in report if d.outdated is False]
    unknown = [d for d in report if d.outdated is None]

    if outdated:
        print(f"WARNING: {len(outdated)} outdated:")
        for d in outdated:
            print(f"   {d.name}: pinned {d.current_specifier!r}, latest is {d.latest}")
    if fresh:
        print(f"OK: {len(fresh)} up to date: {', '.join(d.name for d in fresh)}")
    if unknown:
        print(f"UNKNOWN: {len(unknown)} could not be checked: {', '.join(d.name for d in unknown)}")


report = build_report(deps)
print_report(report)

The sample `pyproject.toml` above deliberately has an intentionally-wrong package name (`not-a-real-package-xyz`) so you can see the "could not be checked" bucket populate too, not just outdated/fresh — a real freshness checker has to handle that gracefully, not crash the whole report over one bad entry.

Try editing `SAMPLE_PYPROJECT` above with dependencies from a real project of yours and re-running the notebook top to bottom.